# Event-TimeRAF Final Results and Figures

This notebook reads one frozen, manifest-backed final run from the main pipeline. It does not train models or alter predictions, and it rejects mixed run identifiers, non-final runs, or runs that did not complete the frozen-TSFM publication gate.

In [ ]:
from pathlib import Path, PurePosixPath
import json
import shutil
import sys
import zipfile
import pandas as pd

RESULTS_ROOT_OVERRIDE = None
RESULTS_RUN_OVERRIDE = None  # Optional exact directory containing logs/run_manifest.json.
REQUIRE_FINAL_PUBLICATION_RUN = True
REQUIRE_PUBLICATION_TITLE_ALLOWED = True

def has_project_files(path):
    return (path / 'configs' / 'default.yaml').exists() and (path / 'src' / 'event_timeraf' / 'config.py').exists()

def reset_directory(path):
    path = path.resolve()
    kaggle_working = Path('/kaggle/working').resolve()
    if path.exists():
        if not path.is_relative_to(kaggle_working):
            raise RuntimeError(f'Refusing to refresh non-working directory: {path}')
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def locate_results_root():
    if RESULTS_ROOT_OVERRIDE is not None:
        override = Path(RESULTS_ROOT_OVERRIDE).resolve()
        if not has_project_files(override):
            raise FileNotFoundError(f'RESULTS_ROOT_OVERRIDE is missing src/event_timeraf/config.py: {override}')
        return override
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, Path('/kaggle/working/event_timeraf')]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(path.parent.parent for path in kaggle_input.rglob('configs/default.yaml'))
    source = next((path for path in candidates if has_project_files(path)), None)
    if source is None and kaggle_input.exists():
        for archive in sorted(kaggle_input.rglob('*.zip')):
            with zipfile.ZipFile(archive) as bundle:
                members = [PurePosixPath(name) for name in bundle.namelist()]
                config_member = next((member for member in members if member.parts[-2:] == ('configs', 'default.yaml')), None)
                package_member = next((member for member in members if member.parts[-3:] == ('src', 'event_timeraf', 'config.py')), None)
                if config_member is None or package_member is None:
                    continue
                if any(member.is_absolute() or '..' in member.parts for member in members):
                    raise RuntimeError(f'Unsafe paths found in attached archive: {archive}')
                extracted = Path('/kaggle/working/event_timeraf_results_source')
                reset_directory(extracted)
                bundle.extractall(extracted)
                source = extracted.joinpath(*config_member.parts[:-2])
                break
    if source is None:
        raise FileNotFoundError('Set RESULTS_ROOT_OVERRIDE to a completed run directory.')
    if kaggle_input.exists() and source.is_relative_to(kaggle_input):
        writable = Path('/kaggle/working/event_timeraf_results')
        writable.mkdir(parents=True, exist_ok=True)
        for directory in ('configs', 'src', 'outputs'):
            if (source / directory).exists():
                target = writable / directory
                if target.exists():
                    shutil.rmtree(target)
                shutil.copytree(source / directory, target)
        return writable
    return source

PROJECT_ROOT = locate_results_root()
PACKAGE_ROOT = PROJECT_ROOT / 'src' / 'event_timeraf'
if not (PACKAGE_ROOT / 'config.py').exists():
    raise FileNotFoundError(f'Package file missing after setup: {PACKAGE_ROOT / "config.py"}')
for module_name in list(sys.modules):
    if module_name == 'event_timeraf' or module_name.startswith('event_timeraf.'):
        del sys.modules[module_name]
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print({'project_root': str(PROJECT_ROOT), 'package_root': str(PACKAGE_ROOT)})
from event_timeraf.config import load_config
from event_timeraf.plots import plot_horizon_metrics

cfg = load_config(PROJECT_ROOT / 'configs' / 'default.yaml', PROJECT_ROOT)
if RESULTS_RUN_OVERRIDE is not None:
    RUN_OUTPUT = Path(RESULTS_RUN_OVERRIDE).resolve()
    manifest_path = RUN_OUTPUT / 'logs' / 'run_manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'No run manifest at {manifest_path}')
else:
    manifest_candidates = list(cfg.paths.outputs.glob('*/logs/run_manifest.json'))
    flat_manifest = cfg.paths.outputs / 'logs' / 'run_manifest.json'
    if flat_manifest.exists():
        manifest_candidates.append(flat_manifest)
    if not manifest_candidates:
        raise FileNotFoundError('No completed run manifest found. Set RESULTS_RUN_OVERRIDE explicitly.')
    manifest_path = max(manifest_candidates, key=lambda candidate: candidate.stat().st_mtime)
    RUN_OUTPUT = manifest_path.parent.parent

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
audit = json.loads((RUN_OUTPUT / 'audit' / 'data_audit.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(RUN_OUTPUT / 'tables' / 'metrics.csv')
predictions = pd.read_parquet(RUN_OUTPUT / 'predictions' / 'predictions.parquet')
explanations = pd.read_parquet(RUN_OUTPUT / 'evidence' / 'explanations.parquet')
ablation = pd.read_csv(RUN_OUTPUT / 'tables' / 'ablation_results.csv')
subset_counts = pd.read_csv(RUN_OUTPUT / 'tables' / 'subset_counts.csv')
drift_results = pd.read_csv(RUN_OUTPUT / 'tables' / 'drift_period_results.csv')
event_results = pd.read_csv(RUN_OUTPUT / 'tables' / 'event_period_results.csv')
run_ids = (
    set(metrics['run_id']) | set(predictions['run_id']) | set(explanations['run_id'])
    | set(ablation['run_id']) | set(subset_counts['run_id'])
    | set(drift_results['run_id']) | set(event_results['run_id'])
)
if run_ids != {manifest['run_id']}:
    raise RuntimeError(f'Mixed or stale run artifacts detected: {sorted(run_ids)} vs manifest {manifest["run_id"]}')
availability_modes = (
    set(metrics['event_availability_mode']) | set(predictions['event_availability_mode'])
    | set(explanations['event_availability_mode'])
)
if availability_modes != {manifest['run_options']['event_availability_mode']}:
    raise RuntimeError(f'Event-availability metadata mismatch: {sorted(availability_modes)}')
run_options = manifest['run_options']
if REQUIRE_FINAL_PUBLICATION_RUN:
    required_true_options = {
        'require_events': run_options.get('require_events'),
        'final_experiment': run_options.get('final_experiment'),
        'retrieval_evidence_reviewed': run_options.get('retrieval_evidence_reviewed'),
        'run_tsf_model': run_options.get('run_tsf_model'),
    }
    failed = [name for name, value in required_true_options.items() if value is not True]
    if failed:
        raise RuntimeError(f'Not a final-publication run; failed manifest options: {failed}')
    if not audit.get('core_ready') or not audit.get('event_ready'):
        raise RuntimeError('Final run failed data-readiness gates in data_audit.json')
if REQUIRE_PUBLICATION_TITLE_ALLOWED and not run_options.get('publication_title_allowed'):
    raise RuntimeError('Frozen-TSFM publication gate did not complete; do not use foundation-model title/claims.')
print({
    'run_output': str(RUN_OUTPUT),
    'run_id': manifest['run_id'],
    'event_availability_mode': run_options['event_availability_mode'],
    'final_experiment': run_options.get('final_experiment'),
    'run_tsf_model': run_options.get('run_tsf_model'),
    'retrieval_evidence_reviewed': run_options.get('retrieval_evidence_reviewed'),
    'publication_title_allowed': run_options.get('publication_title_allowed'),
})


In [ ]:
overall = metrics.loc[
    (metrics['horizon'].astype(str) == 'overall') & (metrics['subset'] == 'all')
    & metrics['model'].str.match(r'^[MC]')
].sort_values('mse')
display(overall)
primary_metrics = metrics.loc[metrics['model'].str.startswith('M')]
plot_horizon_metrics(primary_metrics, 'mse')
plot_horizon_metrics(primary_metrics, 'mae')
display(subset_counts)
display(ablation)


def read_artifact(relative_name):
    matches = [path for path in RUN_OUTPUT.rglob(relative_name) if path.is_file()]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {relative_name} under the selected run, found {matches}')
    return pd.read_csv(matches[0])

display(read_artifact('primary_inference.csv'))
display(read_artifact('kb_stride_sensitivity.csv'))
display(read_artifact('event_weight_sensitivity.csv'))
display(read_artifact('window_attrition.csv'))
display(read_artifact('subset_target_statistics.csv'))

In [ ]:
display(drift_results)
display(event_results)
display(explanations.sort_values('drift_score', ascending=False).head(10))
